In [1]:
!pip install -U docling beautifulsoup4 requests transformers sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.4/451.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 65.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.5 

In [2]:
import json
import re
import os
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

# Config
# NOTE: run this from repo root directory
RULEBOOKS = [
    {
        "path": "/kaggle/input/datasets/saadimam2020/embedding-indexing-fixed/Official-2025-26-NBA-Playing-Rules.pdf", 
        "doc_id": "nba_rulebook_2025",
        "source": "nba_rulebook",
        "source_url": "https://ak-static.cms.nba.com/wp-content/uploads/sites/4/2025/10/Official-2025-26-NBA-Playing-Rules.pdf",
        "league": "NBA",
        # Rules Index: 2-7, Court Diagram: 8, Referee Hand Signals: 72-75
        "skip_pages": set(range(1, 9)) | set(range(72, 77)),
    },
    # Q: do we keep the appendixes in the fiba doc? currently they are still there
    {
        "path": "/kaggle/input/datasets/saadimam2020/embedding-indexing-fixed/fiba-official-rules-2024-v10a.pdf",
        "doc_id": "fiba_rulebook_2024",
        "source": "fiba_rulebook",
        "source_url": "https://www.fiba.basketball/documents/official-basketball-rules",
        "league": "FIBA",
        "skip_pages": set(range(1, 5)) | set(range(7, 8)) | set(range(62, 71)) | set(range(97, 106)),
    }
    
]

# HybridChunker settings (used internally for page-aware filtering only)
TOKENIZER_MODEL = "BAAI/bge-m3"
MAX_TOKENS = 500          # max tokens per chunk — keeps chunks under embedding limit
MIN_SECTION_CHARS = 80    # skip sections shorter than this

# Fixed-size chunking settings
# CHUNK_SIZE is set below the embedding model's 512-token hard limit to leave
# room for the context prefix (league + rulebook label) that gets prepended.
CHUNK_SIZE    = 400   # tokens per chunk
CHUNK_OVERLAP = 50    # sliding-window overlap between consecutive chunks

OUTPUT_DIR = Path("corpus/processed")
OUTPUT_FILE = OUTPUT_DIR / "layer1_rulebook_chunks_fixed.json"

# A single chunk is defined as:

@dataclass
class RulebookChunk:
    chunk_id: str
    doc_id: str
    text: str                    # chunk text WITH context prefix — this gets embedded
    metadata: dict = field(default_factory=dict)


def split_into_fixed_chunks(text: str,tokenizer: AutoTokenizer,
                            chunk_size: int = CHUNK_SIZE,
                            overlap: int = CHUNK_OVERLAP,) -> list[str]:
    """
    Splits *text* into fixed-size token windows with *overlap* tokens.

    Using the BGE-M3 tokenizer, as it is also our embedding model
    """
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    chunks: list[str] = []
    start = 0
    while start < len(token_ids):
        end = min(start + chunk_size, len(token_ids))
        chunk_text = tokenizer.decode(token_ids[start:end], skip_special_tokens=True)
        if chunk_text.strip():
            chunks.append(chunk_text.strip())
        if end >= len(token_ids):
            break
        start += chunk_size - overlap

    return chunks

def parse_heading_path(headings: list[str]) -> dict:
    """
    The rulebooks have defined rules/sections headings which are defined in docling, for extracting in a similar order
    Docling returns headings ordered from outermost to innermost.
    e.g. ["RULE NO. 4—DEFINITIONS", "Section II—Dribble"]
    """
    result = {
        "rule_number": None,
        "rule_title": None,
        "section": None,
        "section_title": None,
        "chunk_type": "general",
    }

    for heading in headings:
        heading_clean = heading.strip()

        # IMPROVED REGEX: Handles RULE NO. 4, RULE 4, hyphens, em-dashes, and colons
        rule_match = re.match(
            r'RULE\s+(?:NO\.?\s+)?(\d+[A-Z]?)\s*[:—\-–\s]\s*(.+)',
            heading_clean, re.IGNORECASE
        )
        if rule_match:
            result["rule_number"] = f"Rule {rule_match.group(1)}"
            result["rule_title"] = rule_match.group(2).strip().title()
            result["chunk_type"] = "rule_section"
            continue

        # IMPROVED REGEX: Handles Section I-Title, Section I: Title, etc.
        section_match = re.match(
            r'Section\s+([IVXLCDM]+)\s*[:—\-–\s]\s*(.+)',
            heading_clean, re.IGNORECASE
        )
        if section_match:
            result["section"] = f"Section {section_match.group(1).upper()}"
            result["section_title"] = section_match.group(2).strip().title()
            continue

        # Comments on the Rules section
        if re.search(r'comments\s+on\s+the\s+rules', heading_clean, re.IGNORECASE):
            result["chunk_type"] = "comments"
            continue

        # Definitions section
        if re.search(r'definitions', heading_clean, re.IGNORECASE):
            result["chunk_type"] = "definitions"
            continue

    return result


def build_context_prefix(heading_info: dict, league: str) -> str:
    """
    Builds a natural-language prefix prepended to every chunk text.
    This dramatically improves BM25 matching on rule-specific queries
    like 'Rule 4 traveling' or 'NBA defensive three seconds'.

    Example output:
      "NBA Rulebook — Rule 10 (Violations And Penalties), Section XIII (Traveling):\n\n"
    """
    parts = [f"{league} Rulebook"]

    if heading_info["rule_number"] and heading_info["rule_title"]:
        parts.append(
            f"{heading_info['rule_number']} ({heading_info['rule_title']})"
        )

    if heading_info["section"] and heading_info["section_title"]:
        parts.append(
            f"{heading_info['section']} ({heading_info['section_title']})"
        )

    prefix = " — ".join(parts) + ":\n\n"
    return prefix


def is_noise_chunk(text: str) -> bool:
    """
    Filters out chunks that are pure navigation or have no retrieval value.
    - Very short chunks (just a heading with no body)
    - Chunks that are entirely page numbers or dashes
    - Chunks that are just "PAGE" markers from the index

    Note that some rulebooks like FIBA have diagrams alongside text, so we need to make sure diagrams don't leave stray text behind
    leading to useless chunks. Docling's OCR is pretty good at ignoring diagrams, but we add this as an extra safety net.
    """
    stripped = text.strip()
    if len(stripped) < MIN_SECTION_CHARS:
        return True

    # Pure page number line  e.g. "- 9 -"
    if re.match(r'^[-\s\d]+$', stripped):
        return True

    # Index entry patterns (just "Rule No. X ... Page Y")
    if re.match(r'^(Section|Rule|PAGE)\s+', stripped) and len(stripped) < 60:
        return True
    # If the text has multiple lines with dot leaders ending in numbers 
    if len(re.findall(r'\.{5,}\s*\d+', stripped)) > 3:
        return True

    return False

def process_rulebook(config: dict, tokenizer: AutoTokenizer) -> list[RulebookChunk]:
    """
    Converts one PDF into fixed-size RulebookChunk objects.

    1. Parse the PDF with Docling (for PDF layout parsing).
    2. Run HybridChunker to obtain page-number metadata per text segment,
       which lets us follow the ``skip_pages``, to skip unwanted text.
    3. After filtering, concatenate the remaining segment texts into one continuous
       string and apply a sliding-window fixed-size tokenizer split.
    4. Each output chunk gets a lightweight context prefix
       (e.g. "NBA Rulebook:") so BM25 can still match on the league name.
    """
    pdf_path   = config["path"]
    doc_id     = config["doc_id"]
    league     = config["league"]
    skip_pages = config.get("skip_pages", set())

    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path}  [fixed-size chunking]")
    print(f"Skipping pages: {sorted(skip_pages)}")

    if not os.path.exists(pdf_path):
        print(f"  [!] File not found: {pdf_path} — skipping.")
        return []

    # Step 1
    pipeline_options = PdfPipelineOptions(
        do_ocr=False,
        do_table_structure=False,
        generate_page_images=False,
    )
    converter = DocumentConverter(
        format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
    )

    print("Parsing PDF with Docling...")
    result = converter.convert(pdf_path)
    doc    = result.document

    # Step 2 
    chunker    = HybridChunker(
        tokenizer=TOKENIZER_MODEL,
        max_tokens=MAX_TOKENS,
        merge_peers=True,
    )
    raw_chunks = list(chunker.chunk(doc))
    print(f"Raw segments from Docling: {len(raw_chunks)}")

    filtered_texts: list[str] = []
    skipped_count = 0

    for raw_chunk in raw_chunks:
        chunk_text = raw_chunk.text.strip()

        # Collect page numbers this segment spans
        page_numbers: list[int] = []
        if hasattr(raw_chunk, "meta") and hasattr(raw_chunk.meta, "doc_items"):
            for item in raw_chunk.meta.doc_items:
                if hasattr(item, "prov"):
                    for prov in item.prov:
                        if hasattr(prov, "page_no"):
                            page_numbers.append(prov.page_no)
        page_numbers = sorted(set(page_numbers))

        # Drop segments whose pages are all in the skip list
        if page_numbers and all(p in skip_pages for p in page_numbers):
            skipped_count += 1
            continue

        # Drop noise segments (very short, pure page numbers, index entries)
        if is_noise_chunk(chunk_text):
            skipped_count += 1
            continue

        filtered_texts.append(chunk_text)

    print(f"  Skipped segments (noise/index/diagrams): {skipped_count}")
    print(f"  Segments kept for chunking: {len(filtered_texts)}")

    if not filtered_texts:
        return []

    # Step 3 
    full_text = "\n\n".join(filtered_texts)
    # Collapse runs of 3+ newlines so the tokenizer doesn't waste tokens on whitespace
    full_text = re.sub(r"\n{3,}", "\n\n", full_text).strip()

    raw_fixed_chunks = split_into_fixed_chunks(full_text, tokenizer, CHUNK_SIZE, CHUNK_OVERLAP)
    print(f"  Fixed-size chunks produced: {len(raw_fixed_chunks)}")

    # Step 4 — Build RulebookChunk objects with metadata
    context_prefix = f"{league} Rulebook:\n\n"

    processed_chunks: list[RulebookChunk] = []
    for i, chunk_text in enumerate(raw_fixed_chunks):
        if is_noise_chunk(chunk_text):
            continue

        enriched_text = context_prefix + chunk_text
        chunk_id      = f"{doc_id}_fschunk_{i:04d}"

        metadata = {
            "source":         config["source"],
            "layer":          1,
            "layer_name":     "rulebook",
            "doc_id":         doc_id,
            "league":         league,
            "source_url":     config["source_url"],
            "chunk_type":     "fixed_size",
            # "chunk_index":    i,
            # "chunk_size":     CHUNK_SIZE,
            # "chunk_overlap":  CHUNK_OVERLAP,
            # Fields kept for schema compatibility with the hybrid version;
            # values are None because fixed-size chunks don't map 1-to-1
            # to a single rule/section boundary.
            "rule_number":    None,
            "rule_title":     None,
            "section":        None,
            "section_title":  None,
            "headings_path":  [],
            "page_numbers":   [],
        }

        processed_chunks.append(RulebookChunk(
            chunk_id=chunk_id,
            doc_id=doc_id,
            text=enriched_text,
            metadata=metadata,
        ))

    print(f"  Final usable fixed-size chunks: {len(processed_chunks)}")
    return processed_chunks

# print a summary of the chunk distribution
def print_chunk_stats(chunks: list[RulebookChunk]) -> None:
    print(f"\n{'='*60}")
    print(f"CHUNK STATISTICS — Total: {len(chunks)}")
    print(f"{'='*60}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print(f"STEP 0: Loading tokenizer ({TOKENIZER_MODEL})")
    print("=" * 60)
    # Load once here; passing it to process_rulebook avoids re-downloading
    # the tokenizer vocabulary for every PDF.
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
    print(f"  Tokenizer ready. Chunk size: {CHUNK_SIZE} tokens, overlap: {CHUNK_OVERLAP} tokens.")

    all_chunks: list[RulebookChunk] = []
    for config in RULEBOOKS:
        chunks = process_rulebook(config, tokenizer)
        all_chunks.extend(chunks)

    print_chunk_stats(all_chunks)
    
    # Serialize to JSON
    output_data = [asdict(chunk) for chunk in all_chunks]
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    print(f"\n✓ Saved {len(all_chunks)} fixed-size chunks to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

STEP 0: Loading tokenizer (BAAI/bge-m3)


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (47432 > 8192). Running this sequence through the model will result in indexing errors


Raw segments from Docling: 195
  Skipped segments (noise/index/diagrams): 11
  Segments kept for chunking: 184
  Fixed-size chunks produced: 136
  Final usable fixed-size chunks: 136

Processing: /kaggle/input/datasets/saadimam2020/embedding-indexing-fixed/fiba-official-rules-2024-v10a.pdf  [fixed-size chunking]
Skipping pages: [1, 2, 3, 4, 7, 62, 63, 64, 65, 66, 67, 68, 69, 70, 97, 98, 99, 100, 101, 102, 103, 104, 105]
Parsing PDF with Docling...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Raw segments from Docling: 244
  Skipped segments (noise/index/diagrams): 44
  Segments kept for chunking: 200
  Fixed-size chunks produced: 109
  Final usable fixed-size chunks: 109

CHUNK STATISTICS — Total: 245

✓ Saved 245 fixed-size chunks to: corpus/processed/layer1_rulebook_chunks_fixed.json


In [3]:
import json
import os
import re
from pathlib import Path
from dataclasses import dataclass, field, asdict
from transformers import AutoTokenizer

# Folder containing your raw scraped JSON files
# Each subfolder (Glossary/, Offense/, Defense/) contains .json files
RAW_INPUT = Path("/kaggle/input/datasets/saadimam2020/embedding-indexing-fixed/layer2_hoopstudent.json")

OUTPUT_DIR  = Path("corpus/processed")
OUTPUT_FILE = OUTPUT_DIR / "layer2_chunks_fixed.json"

# Tokenizer — must match the embedding model used in embedding_indexing.py
TOKENIZER_MODEL = "BAAI/bge-m3"

# Fixed-size chunking settings (mirror the values in preprocess_layer1.py)
CHUNK_SIZE    = 400   # tokens per chunk
CHUNK_OVERLAP = 50    # sliding-window overlap between consecutive chunks

# Minimum character length for a section to be included in the full text
MIN_SECTION_CHARS = 80

# Sections to skip entirely 
SKIP_HEADINGS = {
    "introduction",
    "how to comprehend the player roles and diagrams on this page",
    "how to understand the player roles and diagrams on this page",
    "table of contents",
}

# NOTE: this needs to be the same as layer 1
@dataclass
class HoopStudentChunk:
    chunk_id: str
    doc_id: str
    text: str        # enriched text with context prefix — this gets embedded
    metadata: dict = field(default_factory=dict)

def is_skip_heading(heading: str) -> bool:
    # Returns True if this section heading should be skipped.
    return heading.strip().lower() in SKIP_HEADINGS


def is_noise_section(text: str) -> bool:
    # Returns True if a section's text is too short to be meaningful
    return len(text.strip()) < MIN_SECTION_CHARS


def split_into_fixed_chunks(text: str, tokenizer: AutoTokenizer,
                            chunk_size: int = CHUNK_SIZE,
                            overlap: int = CHUNK_OVERLAP,) -> list[str]:
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    chunks: list[str] = []
    start = 0
    while start < len(token_ids):
        end        = min(start + chunk_size, len(token_ids))
        chunk_text = tokenizer.decode(token_ids[start:end], skip_special_tokens=True)
        if chunk_text.strip():
            chunks.append(chunk_text.strip())
        if end >= len(token_ids):
            break
        start += chunk_size - overlap

    return chunks


def process_document_fixed_size(
    doc: dict,
    tokenizer: AutoTokenizer,
) -> list[HoopStudentChunk]:
    """
    Converts one HoopStudent document into fixed-size token chunks.

    All text for a term (concise definition + every non-noise section) is
    concatenated into a single string in reading order, then split with a
    sliding-window tokenizer. 

    The term name is prepended as a context prefix on every chunk so that
    BM25 can still match an exact term query (e.g. "pick and roll") even when
    the term name itself isn't repeated inside the chunk body.
    """
    term = doc["term"]

    # Step 1 
    parts: list[str] = []

    # Concise definition first — densest signal for exact-match queries
    if doc.get("concise_definition"):
        raw_def = doc["concise_definition"].strip()
        # Strip any redundant "term : " prefix the scraper sometimes includes
        clean_def = re.sub(
            r"^" + re.escape(term) + r"\s*[:\-–]\s*",
            "",
            raw_def,
            flags=re.IGNORECASE,
        ).strip()
        parts.append(f"{term}: {clean_def}")

    # Sections in document order
    for section in doc.get("sections", []):
        heading = section.get("heading", "").strip()
        text    = section.get("text",    "").strip()

        if is_skip_heading(heading) or is_noise_section(text):
            continue

        # Include the section heading
        if heading:
            parts.append(f"{heading}:\n{text}")
        else:
            parts.append(text)

    if not parts:
        return []

    full_text = "\n\n".join(parts)
    # Collapse runs of 3+ newlines to avoid wasting tokens on whitespace
    full_text = re.sub(r"\n{3,}", "\n\n", full_text).strip()

    # Step 2 
    raw_chunks = split_into_fixed_chunks(full_text, tokenizer)

    # Step 3 
    chunks: list[HoopStudentChunk] = []

    for i, chunk_text in enumerate(raw_chunks):
        if len(chunk_text.strip()) < MIN_SECTION_CHARS:
            continue

        # Prepend the term name so every chunk is self-contained for retrieval
        enriched_text = f"{term}:\n\n{chunk_text}"
        chunk_id      = f"{doc['doc_id']}_fschunk_{i:03d}"

        metadata = {
            "source":        "hoopstudent",
            "layer":         2,
            "layer_name":    "plays_and_actions",
            "doc_id":        doc["doc_id"],
            "term":          term,
            "category":      doc.get("category", ""),
            "source_site":   doc.get("source_site", "hoopstudent"),
            "source_url":    doc.get("source_url", ""),
            "chunk_type":    "fixed_size",
            # "chunk_index":   i,
            # "chunk_size":    CHUNK_SIZE,
            # "chunk_overlap": CHUNK_OVERLAP,
            # Kept for schema compatibility with the section-based version
            "section_heading": None,
        }

        chunks.append(HoopStudentChunk(
            chunk_id=chunk_id,
            doc_id=doc["doc_id"],
            text=enriched_text,
            metadata=metadata,
        ))

    return chunks


def load_raw_documents(file_path: Path) -> list[dict]:
    # Loads the single JSON file containing the list of all scraped terms.
    
    if not file_path.exists():
        print(f"[!] Raw data file not found: {file_path}")
        return []

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            documents = json.load(f)
            
        # Ensure it's a list
        if not isinstance(documents, list):
            print(f"[!] Expected a list of documents in {file_path.name}")
            return []

        # Simple validation
        valid_docs = [d for d in documents if "doc_id" in d and "term" in d]
        return valid_docs

    except Exception as e:
        print(f"  [!] Error reading {file_path.name}: {e}")
        return []

# STATS: print a summary after processing
def print_stats(chunks: list[HoopStudentChunk], documents: list[dict]) -> None:
    print(f"\n{'='*60}")
    print(f"LAYER 2 CHUNK STATISTICS")
    print(f"{'='*60}")
    print(f"Documents processed : {len(documents)}")
    print(f"Total chunks created: {len(chunks)}")

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print("LAYER 2 PREPROCESSING — HoopStudent  [fixed-size chunking]")
    print("=" * 60)
    print(f"Input file  : {RAW_INPUT}")
    print(f"Output file : {OUTPUT_FILE}")

    # Load tokenizer once
    print(f"\nLoading tokenizer ({TOKENIZER_MODEL})...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
    print(f"  Tokenizer ready. Chunk size: {CHUNK_SIZE} tokens, overlap: {CHUNK_OVERLAP} tokens.")

    # Step 1: Load the single raw JSON file
    print(f"\nLoading raw data...")
    documents = load_raw_documents(RAW_INPUT)

    if not documents:
        print("\n[!] No documents found.")
        return

    print(f"Loaded {len(documents)} documents.")

    # Step 2: Process each document into fixed-size chunks
    print("\nChunking documents (fixed-size)...")
    all_chunks: list[HoopStudentChunk] = []

    for doc in documents:
        doc_chunks = process_document_fixed_size(doc, tokenizer)
        all_chunks.extend(doc_chunks)

    # Step 3: Print stats and sample output
    print_stats(all_chunks, documents)

    # Step 4: Save to JSON — same format as layer 1
    output_data = [asdict(chunk) for chunk in all_chunks]

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    print(f"\n✓ Saved {len(all_chunks)} fixed-size chunks to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

LAYER 2 PREPROCESSING — HoopStudent  [fixed-size chunking]
Input file  : /kaggle/input/datasets/saadimam2020/embedding-indexing-fixed/layer2_hoopstudent.json
Output file : corpus/processed/layer2_chunks_fixed.json

Loading tokenizer (BAAI/bge-m3)...
  Tokenizer ready. Chunk size: 400 tokens, overlap: 50 tokens.

Loading raw data...
Loaded 351 documents.

Chunking documents (fixed-size)...


Token indices sequence length is longer than the specified maximum sequence length for this model (8618 > 8192). Running this sequence through the model will result in indexing errors



LAYER 2 CHUNK STATISTICS
Documents processed : 351
Total chunks created: 3006

✓ Saved 3006 fixed-size chunks to: corpus/processed/layer2_chunks_fixed.json
